# 09 Cue-Feature Counts

Reads `processed/08_words.csv` and counts, for every cue, how many people listed each word. **Nothing is left out**: every cue-word pair is in the file, including words only one person wrote. Filtering (for example the 16% or top-five rule from the 2019 paper) is left for later.

| Column | Meaning |
|---|---|
| `cue` | the cue word |
| `word` | the feature word (lemma, after all processing) |
| `count` | number of people who listed this word for this cue |
| `n_final` | people with at least one word for this cue |
| `n_answered` | people who answered this cue before any exclusions |
| `prop_final` | `count / n_final` |
| `prop_answered` | `count / n_answered` |
| `pos` | most common part of speech for this word under this cue |
| `mean_position` | average position in people's lists (1 = first word listed) |
| `original_words` | the forms people actually wrote, e.g. `leg|legs` |

Output: `processed/09_cue_feature_counts.csv`.

In [1]:
import pandas as pd

In [2]:
words = pd.read_csv("processed/08_words.csv", dtype={"subject_id": str, "position": int,
                                                       "n_answered": int, "n_final": int}, keep_default_na=False)
print(f"{len(words):,} rows, {words['cue'].nunique():,} cues")

952,792 rows, 3,722 cues


In [3]:
grouped = words.groupby(["cue", "word"])

counts = grouped.agg(
    count=("subject_id", "nunique"),
    n_final=("n_final", "first"),
    n_answered=("n_answered", "first"),
    mean_position=("position", "mean"),
).reset_index()

counts["prop_final"] = counts["count"] / counts["n_final"]
counts["prop_answered"] = counts["count"] / counts["n_answered"]
counts["mean_position"] = counts["mean_position"].round(2)

counts["pos"] = grouped["pos"].agg(lambda s: s.value_counts().index[0]).values
counts["original_words"] = grouped["original_word"].agg(
    lambda s: "|".join(sorted(set(s.str.lower())))
).values

counts = counts[["cue", "word", "count", "n_final", "n_answered", "prop_final", "prop_answered",
                 "pos", "mean_position", "original_words"]]
counts = counts.sort_values(["cue", "count", "word"], ascending=[True, False, True])
counts["prop_final"] = counts["prop_final"].round(4)
counts["prop_answered"] = counts["prop_answered"].round(4)
print(f"{len(counts):,} cue-word pairs")

429,486 cue-word pairs


## Checks

In [4]:
# a word cannot be listed by more people than answered the cue
assert (counts["count"] <= counts["n_final"]).all()
print("max count > n_final:", (counts["count"] > counts["n_final"]).sum())
print(f"pairs listed by one person: {(counts['count'] == 1).mean():.0%}")
counts[counts["cue"] == "sidewalk"].head(12)

max count > n_final: 0
pairs listed by one person: 68%


,cue,word,count,n_final,n_answered,prop_final,prop_answered,pos,mean_position,original_words
341301,sidewalk,concrete,19,30,30,0.6333,0.6333,NOUN,3.89,concrete
341384,sidewalk,walk,14,30,30,0.4667,0.4667,VERB,2.71,walk|walked|walking
341347,sidewalk,pedestrian,10,30,30,0.3333,0.3333,NOUN,3.10,pedestrian|pedestrians
341355,sidewalk,road,10,30,30,0.3333,0.3333,NOUN,4.60,road|roads
341344,sidewalk,path,7,30,30,0.2333,0.2333,NOUN,2.86,path
341372,sidewalk,street,6,30,30,0.2000,0.2000,NOUN,5.33,street|streets
341389,sidewalk,way,5,30,30,0.1667,0.1667,NOUN,4.20,way|ways
341290,sidewalk,bike,4,30,30,0.1333,0.1333,NOUN,3.50,bike|bikes|biking
341299,sidewalk,color,4,30,30,0.1333,0.1333,NOUN,9.75,color
341339,sidewalk,next,4,30,30,0.1333,0.1333,ADV,2.75,next


In [5]:
counts.to_csv("processed/09_cue_feature_counts.csv", index=False)
print(f"Saved {len(counts):,} rows to processed/09_cue_feature_counts.csv")

Saved 429,486 rows to processed/09_cue_feature_counts.csv
